In [ ]:
from __future__ import annotations

import os
import tempfile
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq


def find_problematic_parquet_columns(
    parquet_path: str | Path,
) -> tuple[list[str], list[str]]:
    """Find readable and unreadable Parquet columns.

    Args:
        parquet_path: Path to the Parquet file.

    Returns:
        A tuple of readable columns and problematic columns.
    """
    parquet_path = Path(parquet_path)
    parquet_file = pq.ParquetFile(parquet_path)

    readable_columns: list[str] = []
    problematic_columns: list[str] = []

    for column_name in parquet_file.schema_arrow.names:
        try:
            for row_group_index in range(parquet_file.num_row_groups):
                parquet_file.read_row_group(
                    row_group_index,
                    columns=[column_name],
                    use_threads=False,
                )

            readable_columns.append(column_name)
            print(f"OK: {column_name}")

        except Exception as error:
            problematic_columns.append(column_name)
            print(
                f"FAILED: {column_name} | "
                f"{type(error).__name__}: {error}"
            )

    return readable_columns, problematic_columns


def remove_problematic_parquet_columns(
    parquet_path: str | Path,
    compression: str = "gzip",
) -> list[str]:
    """Remove unreadable columns from a Parquet file.

    The readable columns are copied row group by row group to a temporary file.
    The original file is preserved as a backup.

    Args:
        parquet_path: Path to the damaged Parquet file.
        compression: Compression codec for the repaired file.

    Returns:
        Names of the removed columns.

    Raises:
        RuntimeError: If the file cannot be repaired.
    """
    parquet_path = Path(parquet_path)

    readable_columns, problematic_columns = (
        find_problematic_parquet_columns(parquet_path)
    )

    if not problematic_columns:
        print("No problematic columns found.")
        return []

    print(f"Removing columns: {problematic_columns}")

    parquet_file = pq.ParquetFile(parquet_path)

    fd, tmp_name = tempfile.mkstemp(
        dir=parquet_path.parent,
        prefix=parquet_path.name + ".",
        suffix=".repair.tmp",
    )
    os.close(fd)

    tmp_path = Path(tmp_name)
    writer: pq.ParquetWriter | None = None

    try:
        for row_group_index in range(parquet_file.num_row_groups):
            table = parquet_file.read_row_group(
                row_group_index,
                columns=readable_columns,
                use_threads=False,
            )

            if writer is None:
                writer = pq.ParquetWriter(
                    tmp_path,
                    table.schema,
                    compression=compression,
                )

            writer.write_table(table)

        if writer is not None:
            writer.close()
            writer = None

        # Verify that pandas can read the repaired file.
        repaired_df = pd.read_parquet(tmp_path)

        if len(repaired_df) != parquet_file.metadata.num_rows:
            raise RuntimeError(
                "Row count changed during repair."
            )
        
        backup_path = parquet_path.with_name(
            parquet_path.name.replace(
                ".gzip.parquet.backup",
                "_BACKUP.gzip.parquet",
            )
        )

        if backup_path.exists():
            backup_path.unlink()

        os.replace(parquet_path, backup_path)
        os.replace(tmp_path, parquet_path)

        print(f"Repaired file: {parquet_path}")
        print(f"Backup file: {backup_path}")
        print(f"Removed columns: {problematic_columns}")

        return problematic_columns

    except Exception as error:
        if writer is not None:
            writer.close()

        raise RuntimeError(
            f"Failed to repair Parquet file: {parquet_path}"
        ) from error

    finally:
        if tmp_path.exists():
            tmp_path.unlink(missing_ok=True)

In [14]:
# parquet_path = Path("/home/amitner/Backbone-Graph/results/Labeled_Datasets/UAE/UAE_part_all/step_2_BERTopic_clustering/UAE_part_all_step_2.gzip.parquet")
parquet_path = Path("/home/amitner/Backbone-Graph/results/Labeled_Datasets/Venezuela/Venezuela_part_all/step_2_BERTopic_clustering/Venezuela_part_all_step_2.gzip.parquet")
removed_columns = remove_problematic_parquet_columns(parquet_path)

print("Removed columns:", removed_columns)

OK: translated_post
OK: accountid
OK: post_time
OK: is_control
OK: BERTopic_topic_128
OK: BERTopic_prob_128
OK: BERTopic_topic_128_top_words
OK: BERTopic_topic_512
OK: BERTopic_prob_512
OK: BERTopic_topic_512_top_words
OK: BERTopic_topic_256
OK: BERTopic_prob_256
OK: BERTopic_topic_256_top_words
OK: posting_same_topic_BERTopic_topic_128_12H_accountids
FAILED: posting_same_topic_BERTopic_topic_128_3D_accountids | ArrowNotImplementedError: Nested data conversions not implemented for chunked array outputs
Removing columns: ['posting_same_topic_BERTopic_topic_128_3D_accountids']
Repaired file: /home/amitner/Backbone-Graph/results/Labeled_Datasets/Venezuela/Venezuela_part_all/step_2_BERTopic_clustering/Venezuela_part_all_step_2.gzip.parquet
Backup file: /home/amitner/Backbone-Graph/results/Labeled_Datasets/Venezuela/Venezuela_part_all/step_2_BERTopic_clustering/Venezuela_part_all_step_2.gzip.parquet.backup
Removed columns: ['posting_same_topic_BERTopic_topic_128_3D_accountids']
Removed colu

In [ ]:
df = pd.read_parquet(parquet_path)
print(df.shape)
print(df.columns.tolist())

(1705178, 8)
['translated_post', 'accountid', 'post_time', 'is_control', 'BERTopic_topic_10', 'BERTopic_prob_10', 'BERTopic_topic_10_top_words', 'posting_same_topic_BERTopic_topic_10_12H_accountids']
